# 03 Build Silver Tables

Transform Bronze MMSDM rows into typed Silver tables at stable grains for price, demand, regional dispatch, interconnector flows, and generation where source columns are available.


## Configure Silver Run

This cell detects local versus Fabric runtime and defines run parameters. Spark transformations run only in Fabric.


In [1]:
# Cell purpose: Configure Silver Transformation Run.
from pathlib import Path
import importlib
import os
import sys
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())
print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")


run_id=f5b922a9-1270-41fc-8135-23ea366c9d53
runtime=local


## Resolve Runtime Paths and Imports

This cell resolves package paths and imports Spark helpers only in Fabric.


In [2]:
# Cell purpose: Resolve Runtime Paths and Imports.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


repo_root = None
local_output_root = None
package_paths = []
# Cell purpose: Build Silver price and demand tables.
if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / "data"
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))

# Cell purpose: Build Silver interconnector flow table.
if is_local_run:
    import pandas as pd

    from nem_fabric.common_transformations import (
        build_silver_cumulative_price,
        build_silver_generation_by_unit,
        build_silver_predispatch_forecast,
        build_silver_interconnector_flows,
        build_silver_price_demand,
        build_silver_rooftop_pv,
        build_silver_seven_day_outlook,
    )
    from nem_fabric.local_ingestion import append_parquet_rows, read_parquet_table
else:
    F = importlib.import_module("pyspark.sql.functions")
    Window = importlib.import_module("pyspark.sql.window").Window

print(
    "Python search paths added:", [str(path) for path in package_paths if path.exists()]
)
# Cell purpose: Build Silver generation-by-unit table when source columns are available.
if is_local_run:
    print(f"Local output root: {local_output_root}")


Python search paths added: ['c:\\Users\\brcol\\My Drive\\Documents\\!!!Resume\\Sample Work\\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\\src']
Local output root: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data


## Define Shared Silver Helpers

This cell defines NEM region mapping, table checks, and region-name enrichment. It also confirms that Bronze data exists.


In [3]:
# Cell purpose: Define Shared Silver Helpers.
BRONZE_SOURCE_LABELS = [
    "dispatchis",
    "predispatchis",
    "trading_cumulative_price",
    "seven_day_outlook",
    "dispatch_scada",
    "rooftop_pv_actual",
    "next_day_dispatch",
]

if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")

    bronze_dfs = []
    for source_label in BRONZE_SOURCE_LABELS:
        bronze_parquet_path = local_output_root / "tables" / f"bronze_{source_label}"
        if bronze_parquet_path.exists():
            bronze_dfs.append(
                read_parquet_table(bronze_parquet_path).astype(str).fillna("")
            )
    if not bronze_dfs:
        raise RuntimeError("No local Bronze source CSVs exist. Run notebook 02 first.")
    bronze_pdf = pd.concat(bronze_dfs, ignore_index=True, sort=False).fillna("")
    print(f"Local Bronze rows loaded: {len(bronze_pdf)}")
else:
    REGION_MAP = {
        "QLD1": "Queensland",
        "NSW1": "New South Wales",
        "VIC1": "Victoria",
        "SA1": "South Australia",
        "TAS1": "Tasmania",
    }


    def table_exists(table_name: str) -> bool:
        """Return True when a Lakehouse table exists in the current Spark catalogue."""
        return spark.catalog.tableExists(table_name)


    def with_region_name(df, region_col="region"):
        """Add a business-friendly region name using Spark expressions."""
        mapping_expr = F.create_map([item for pair in REGION_MAP.items() for item in (F.lit(pair[0]), F.lit(pair[1]))])
        return df.withColumn("region_name", mapping_expr[F.col(region_col)])


    bronze_tables = [
        spark.table(f"bronze_{source_label}")
        for source_label in BRONZE_SOURCE_LABELS
        if table_exists(f"bronze_{source_label}")
    ]
    if not bronze_tables:
        raise RuntimeError("No Bronze source tables exist. Run notebook 02 first.")
    bronze_sdf = bronze_tables[0]
    for source_table in bronze_tables[1:]:
        bronze_sdf = bronze_sdf.unionByName(source_table, allowMissingColumns=True)


Local Bronze rows loaded: 215630


## Build Silver Price and Demand

This cell joins Dispatch `PRICE` and `REGIONSUM` rows into the main 5-minute regional grain and writes the core Silver tables.


In [4]:
# Cell purpose: Build Silver price and demand tables.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    silver_price_demand = build_silver_price_demand(bronze_pdf, run_id=run_id)
    if silver_price_demand.empty:
        print("No local Silver price/demand rows produced.")
    else:
        records = silver_price_demand.astype(str).to_dict("records")
        append_parquet_rows(
            local_output_root / "tables" / "silver_price_demand_5min.parquet", records
        )
        append_parquet_rows(
            local_output_root / "tables" / "silver_regional_dispatch.parquet", records
        )
        print(f"Local Silver price/demand rows written: {len(silver_price_demand)}")
else:
    # PRICE and REGIONSUM rows share interval and region keys. Joining them creates the main 5-minute price/demand grain.
    price = bronze_sdf.filter(
        (F.col("package_name") == "DISPATCH") & (F.col("table_name") == "PRICE")
    ).select(
        F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
            "settlement_datetime"
        ),
        F.upper(F.col("regionid")).alias("region"),
        F.col("intervention").cast("int").alias("intervention"),
        F.col("rrp").cast("double").alias("price_aud_mwh"),
        F.col("source_url"),
        F.col("source_zip_name"),
        F.col("row_hash").alias("price_row_hash"),
    )

    regionsum = bronze_sdf.filter(
        (F.col("package_name") == "DISPATCH") & (F.col("table_name") == "REGIONSUM")
    ).select(
        F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
            "settlement_datetime"
        ),
        F.upper(F.col("regionid")).alias("region"),
        F.col("intervention").cast("int").alias("intervention"),
        F.col("totaldemand").cast("double").alias("demand_mw"),
        F.col("availablegeneration").cast("double").alias("available_generation_mw"),
        F.col("availableload").cast("double").alias("available_load_mw"),
        F.col("demandforecast").cast("double").alias("demand_forecast_mw"),
        F.col("dispatchablegeneration")
        .cast("double")
        .alias("dispatchable_generation_mw"),
        F.col("dispatchableload").cast("double").alias("dispatchable_load_mw"),
        F.col("netinterchange").cast("double").alias("net_interchange_mw"),
        F.col("excessgeneration").cast("double").alias("excess_generation_mw"),
        F.col("clearedsupply").cast("double").alias("cleared_supply_mw"),
        F.col("semischedule_clearedmw")
        .cast("double")
        .alias("semi_scheduled_generation_mw"),
        (
            F.col("dispatchablegeneration").cast("double")
            - F.col("semischedule_clearedmw").cast("double")
        ).alias("scheduled_generation_mw"),
        F.col("row_hash").alias("regionsum_row_hash"),
    )

    silver_price_demand = price.join(
        regionsum, ["settlement_datetime", "region", "intervention"], "left"
    )
    silver_price_demand = with_region_name(silver_price_demand)
    silver_price_demand = (
        silver_price_demand.withColumn("trading_date", F.to_date("settlement_datetime"))
        .withColumn("year", F.year("settlement_datetime"))
        .withColumn("month", F.month("settlement_datetime"))
        .withColumn("day", F.dayofmonth("settlement_datetime"))
        .withColumn("interval_hour", F.hour("settlement_datetime"))
        .withColumn("interval_minute", F.minute("settlement_datetime"))
        .withColumn("silver_loaded_datetime", F.current_timestamp())
        .withColumn("run_id", F.lit(run_id))
        .dropDuplicates(["settlement_datetime", "region", "intervention"])
    )

    silver_price_demand.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("silver_price_demand_5min")
    silver_price_demand.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("silver_regional_dispatch")
    display(silver_price_demand.orderBy(F.col("settlement_datetime").desc()).limit(20))


Local Silver price/demand rows written: 965


## Build Silver Interconnector Flows

This cell extracts interconnector flow records at interval and interconnector grain when Dispatch interconnector rows are available.


In [5]:
# Cell purpose: Build Silver interconnector flow table.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    interconnector = build_silver_interconnector_flows(bronze_pdf, run_id=run_id)
    if interconnector.empty:
        print("No local interconnector rows available.")
    else:
        append_parquet_rows(
            local_output_root / "tables" / "silver_interconnector_flows.parquet",
            interconnector.astype(str).to_dict("records"),
        )
        print(f"Local Silver interconnector rows written: {len(interconnector)}")
else:
    # Interconnector grain: settlement interval x interconnector x intervention.
    interconnector = (
        bronze_sdf.filter(
            (F.col("package_name") == "DISPATCH")
            & (F.col("table_name") == "INTERCONNECTORRES")
        )
        .select(
            F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                "settlement_datetime"
            ),
            F.col("interconnectorid").alias("interconnector_id"),
            F.col("intervention").cast("int").alias("intervention"),
            F.col("meteredmwflow").cast("double").alias("metered_flow_mw"),
            F.col("mwflow").cast("double").alias("flow_mw"),
            F.col("mwlosses").cast("double").alias("losses_mw"),
            F.col("marginalvalue").cast("double").alias("marginal_value"),
            F.col("exportlimit").cast("double").alias("export_limit_mw"),
            F.col("importlimit").cast("double").alias("import_limit_mw"),
            F.to_date(F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")).alias(
                "trading_date"
            ),
            F.current_timestamp().alias("silver_loaded_datetime"),
            F.lit(run_id).alias("run_id"),
        )
        .dropDuplicates(["settlement_datetime", "interconnector_id", "intervention"])
    )

    if interconnector.limit(1).count() > 0:
        interconnector.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).partitionBy("trading_date").saveAsTable("silver_interconnector_flows")
        display(interconnector.orderBy(F.col("settlement_datetime").desc()).limit(20))
    else:
        print("No interconnector rows available.")


Local Silver interconnector rows written: 1158


## Build Optional Silver Generation

This cell creates a generation-by-unit Silver table only if suitable DUID and generation columns exist in the Bronze source data.


In [6]:
# Cell purpose: Build Silver generation-by-unit table when source columns are available.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    generation = build_silver_generation_by_unit(bronze_pdf, run_id=run_id)
    if generation.empty:
        print("Generation-by-unit columns not available in current local Bronze data.")
    else:
        append_parquet_rows(
            local_output_root / "tables" / "silver_generation_by_unit.parquet",
            generation.astype(str).to_dict("records"),
        )
        print(f"Local Silver generation rows written: {len(generation)}")
else:
    # Unit generation is source-dependent. This creates a Silver table only when DUID and generation-like columns are present.
    candidate_columns = set(bronze_sdf.columns)
    generation_column = next(
        (
            column
            for column in ["dispatchablegeneration", "scadavalue", "generation_mw"]
            if column in candidate_columns
        ),
        None,
    )
    if "duid" in candidate_columns and generation_column:
        generation = (
            bronze_sdf.filter(F.col("duid") != "")
            .select(
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                    "settlement_datetime"
                ),
                F.col("duid"),
                F.col(generation_column).cast("double").alias("generation_mw"),
                F.to_date(
                    F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")
                ).alias("trading_date"),
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
            .dropDuplicates(["settlement_datetime", "duid"])
        )
        generation.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).partitionBy("trading_date").saveAsTable("silver_generation_by_unit")
    else:
        print("Generation-by-unit columns not available in current Bronze data.")


Local Silver generation rows written: 8575


## Build Silver Forecast and Outlook Tables

This cell processes newly enabled forecast and outlook sources into typed Silver tables for downstream Gold dashboard tables.


In [7]:
# Cell purpose: Build Silver forecast, cumulative price, seven-day outlook, and rooftop PV tables.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    tables_root = local_output_root / "tables"
    optional_builders = [
        ("silver_predispatch_forecast", build_silver_predispatch_forecast),
        ("silver_cumulative_price", build_silver_cumulative_price),
        ("silver_seven_day_outlook", build_silver_seven_day_outlook),
        ("silver_rooftop_pv", build_silver_rooftop_pv),
    ]
    for table_name, builder in optional_builders:
        df = builder(bronze_pdf, run_id=run_id)
        if df.empty:
            print(f"No local rows produced for {table_name}.")
            continue
        append_parquet_rows(
            tables_root / f"{table_name}.parquet",
            df.astype(str).to_dict("records"),
        )
        print(f"Local {table_name} rows written: {len(df)}")
else:
    candidate_columns = set(bronze_sdf.columns)

    predispatch = bronze_sdf.filter(F.col("source_folder").contains("Predispatch"))
    if predispatch.limit(1).count() > 0 and "regionid" in candidate_columns:
        price_candidates = [
            column
            for column in ["rrp", "price", "forecast_price"]
            if column in candidate_columns
        ]
        demand_candidates = [
            column
            for column in ["demand", "totaldemand", "demandforecast", "forecast_demand"]
            if column in candidate_columns
        ]
        forecast_price = (
            F.coalesce(*[F.col(column).cast("double") for column in price_candidates])
            if price_candidates
            else F.lit(None).cast("double")
        )
        forecast_demand = (
            F.coalesce(*[F.col(column).cast("double") for column in demand_candidates])
            if demand_candidates
            else F.lit(None).cast("double")
        )
        predispatch_silver = with_region_name(
            predispatch.select(
                F.to_timestamp("file_datetime").alias("forecast_run_datetime"),
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                    "forecast_settlement_datetime"
                ),
                F.upper(F.col("regionid")).alias("region"),
                forecast_price.alias("forecast_price_aud_mwh"),
                forecast_demand.alias("forecast_demand_mw"),
                "source_url",
                "source_zip_name",
                "row_hash",
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
        )
        predispatch_silver = (
            predispatch_silver.filter(F.col("forecast_settlement_datetime").isNotNull())
            .filter(F.col("region").isNotNull() & (F.col("region") != ""))
            .filter(
                F.col("forecast_price_aud_mwh").isNotNull()
                | F.col("forecast_demand_mw").isNotNull()
            )
            .dropDuplicates(
                ["forecast_run_datetime", "forecast_settlement_datetime", "region"]
            )
        )
        predispatch_silver.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable("silver_predispatch_forecast")

    cumulative_rows = bronze_sdf.filter(
        F.col("source_folder").contains("Trading_Cumulative_Price")
    )
    cumulative_column = next(
        (
            column
            for column in [
                "cumulativeprice",
                "cumul_price",
                "cumulative_price",
                "periodcumulativeprice",
            ]
            if column in candidate_columns
        ),
        None,
    )
    if (
        cumulative_column
        and "regionid" in candidate_columns
        and cumulative_rows.limit(1).count() > 0
    ):
        cumulative_silver = with_region_name(
            cumulative_rows.select(
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                    "settlement_datetime"
                ),
                F.to_date(
                    F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")
                ).alias("trading_date"),
                F.upper(F.col("regionid")).alias("region"),
                F.col(cumulative_column)
                .cast("double")
                .alias("cumulative_price_aud_mwh"),
                (
                    F.when(F.col("apcflag") == "1", "Active")
                    .otherwise("Inactive")
                    .alias("administered_price_cap_status")
                    if "apcflag" in candidate_columns
                    else F.lit("Unknown").alias("administered_price_cap_status")
                ),
                "source_url",
                "source_zip_name",
                "row_hash",
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
        ).dropDuplicates(["settlement_datetime", "region"])
        cumulative_silver.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).partitionBy("trading_date").saveAsTable("silver_cumulative_price")

    seven_day_rows = bronze_sdf.filter(
        F.col("source_folder").contains("Seven_Day_Outlook")
    )
    if seven_day_rows.limit(1).count() > 0 and "regionid" in candidate_columns:

        def metric_expr(candidates):
            column = next(
                (
                    candidate
                    for candidate in candidates
                    if candidate in candidate_columns
                ),
                None,
            )
            return (
                F.col(column).cast("double") if column else F.lit(None).cast("double")
            )

        seven_day_silver = with_region_name(
            seven_day_rows.select(
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                    "settlement_datetime"
                ),
                F.to_date(
                    F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")
                ).alias("outlook_date"),
                F.upper(F.col("regionid")).alias("region"),
                metric_expr(
                    ["scheduleddemand", "demand", "demand10", "maximumdemand"]
                ).alias("scheduled_demand_mw"),
                metric_expr(
                    ["scheduledcapacity", "capacity", "availablegeneration"]
                ).alias("scheduled_capacity_mw"),
                metric_expr(
                    ["scheduledreserve", "reserve", "reserverequirement"]
                ).alias("scheduled_reserve_mw"),
                metric_expr(["netinterchange", "interchange"]).alias(
                    "net_interchange_mw"
                ),
                "source_url",
                "source_zip_name",
                "row_hash",
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
        ).dropDuplicates(["settlement_datetime", "region"])
        seven_day_silver.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable("silver_seven_day_outlook")

    rooftop_rows = bronze_sdf.filter(
        F.col("source_folder").contains("Intermittent_Generation")
    )
    rooftop_column = next(
        (
            column
            for column in [
                "power",
                "rooftoppv",
                "rooftop_pv",
                "measurement",
                "scadavalue",
                "actualmw",
            ]
            if column in candidate_columns
        ),
        None,
    )
    if (
        rooftop_column
        and "regionid" in candidate_columns
        and rooftop_rows.limit(1).count() > 0
    ):
        rooftop_silver = with_region_name(
            rooftop_rows.select(
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias(
                    "settlement_datetime"
                ),
                F.to_date(
                    F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")
                ).alias("trading_date"),
                F.upper(F.col("regionid")).alias("region"),
                F.col(rooftop_column).cast("double").alias("rooftop_pv_mw"),
                "source_url",
                "source_zip_name",
                "row_hash",
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
        ).dropDuplicates(["settlement_datetime", "region"])
        rooftop_silver.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).partitionBy("trading_date").saveAsTable("silver_rooftop_pv")


No local rows produced for silver_predispatch_forecast.
No local rows produced for silver_cumulative_price.
No local rows produced for silver_seven_day_outlook.
No local rows produced for silver_rooftop_pv.
